# 입출력과 출처 검증

모델 호출 앞뒤에서 질문과 출처 형식을 파이썬으로 검사합니다.

## 개념을 코드에 연결하기

**호출 표기와 실행 시점.** 입력 검증은 모델 호출 전에 질문의 빈값·길이·허용 문자를 확인합니다. 출력 검증은 모델 답에서 출처 태그를 추출해 이번 검색 결과의 doc_id와 page 조합에 포함되는지 대조합니다. 함수가 반환한 검증 결과와 print로 보이는 화면 문자열은 구분해 읽습니다.

**다른 상황에 적용.** 의료 안내에서는 질문 길이와 금지 입력을 먼저 검사하고, 답변의 약품 코드가 검색된 공식 문서 목록에 있는지 후처리할 수 있습니다. 도메인이 달라져도 호출 전 입력 경계와 호출 후 허용 목록 경계는 분리됩니다.

**비교와 경계.** 형식이 맞는 출처가 실제 주장을 뒷받침한다는 뜻은 아닙니다. 이 검사는 존재하지 않는 ID를 막지만 인용 문장과 근거 본문의 의미 일치까지 판단하지 않습니다. 근거성 평가는 Ch15의 평가와 함께 사용합니다.

**이 셀의 코드 읽기.** 이 셀은 앞 셀에서 만든 값을 입력으로 읽어 위에서 아래 순서로 처리하고, 왼쪽 변수에 다음 셀이 사용할 결과를 저장합니다.


In [1]:
%pip install -q langchain==1.4.0 langchain-openai==1.6.1 python-dotenv==1.2.3

Note: you may need to restart the kernel to use updated packages.


## 실행 환경 준비

LangChain 1.4와 OpenAI 통합 버전을 고정하고 실제 API를 호출합니다. 예상 비용은 0.01달러 미만입니다.

**이 셀의 코드 읽기.** `for` 블록은 입력 항목을 하나씩 처리하며, 반복이 끝난 뒤 모은 값이 다음 셀의 입력이 됩니다. `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다. `assert`는 조건이 참이면 출력 없이 통과하고 거짓이면 실행을 중단하므로, 바로 앞 결과를 검증하는 역할을 합니다.


In [2]:
from pathlib import Path
import os, sys
from dotenv import find_dotenv, load_dotenv

start = Path.cwd().resolve()
candidates = (start, *start.parents)
ROOT = next(path for path in candidates if (path / "labs/part4/ch16-guardrails").is_dir())
LAB = ROOT / "labs/part4/ch16-guardrails"
load_dotenv(find_dotenv(usecwd=True))
assert os.getenv("OPENAI_API_KEY"), "프로젝트 루트 .env에 OPENAI_API_KEY가 필요합니다."
print("Python:", sys.version.split()[0], "| API 키 준비: True")

Python: 3.12.13 | API 키 준비: True


### 다음 코드 셀 읽기

**이 셀의 코드 읽기.** `init_chat_model(...)`은 채팅 모델 객체를 만들며, 이 줄에서는 아직 답변을 생성하지 않습니다. `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다.


In [3]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    "openai:gpt-5.4-mini", temperature=0, max_tokens=140,
    timeout=30, max_retries=0,
)
print("모델 준비:", type(model).__name__)

모델 준비: ChatOpenAI


## 기본 실행

Ch 9의 `[doc_id p.쪽]` 형식을 그대로 사용합니다.

**이 셀의 코드 읽기.** `validate_query(query: str, max_chars: int = 500)`, `validate_answer(answer: str, allowed: set[tuple[str, int]])`는 함수를 정의하는 표기입니다. 이 셀에서는 함수 객체만 준비되고 들여쓴 본문은 뒤에서 함수를 호출할 때 실행됩니다. `return` 뒤의 값은 화면 출력이 아니라 호출한 코드로 전달되는 반환값입니다. `invoke(input)`가 나타나는 줄에서 실제 그래프 또는 모델 실행이 시작되고 반환값이 왼쪽 변수에 저장됩니다. `re.findall(pattern, text)`는 패턴과 일치한 항목 목록을 반환합니다. `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다.


In [4]:
import re

def validate_query(query: str, max_chars: int = 500) -> str:
    normalized = " ".join(query.split())
    if not normalized or len(normalized) > max_chars:
        raise ValueError("질문의 내용과 길이를 확인하세요.")
    return normalized

def validate_answer(answer: str, allowed: set[tuple[str, int]]) -> list[str]:
    citations = [(doc, int(page)) for doc, page in re.findall(r"\[([A-Z]\d{2}) p\.(\d+)\]", answer)]
    if not citations or any(item not in allowed for item in citations):
        raise ValueError("출처 허용 목록을 확인하세요.")
    return [f"[{doc} p.{page}]" for doc, page in citations]

question = validate_query("  도서관 휴관일은 언제인가요?  ")
context = "E01 1쪽: 미르시립도서관은 매주 월요일에 휴관합니다."
messages = [
    {"role": "system", "content": "문서만 사용하고 끝에 [E01 p.1]을 적으세요."},
    {"role": "user", "content": f"질문: {question}\n문서: {context}"},
]
answer = model.invoke(messages)
print(answer.text)
print("검증 출처:", validate_answer(answer.text, {("E01", 1)}))

미르시립도서관은 매주 월요일에 휴관합니다. [E01 p.1]
검증 출처: ['[E01 p.1]']


## 중간 결과 관찰

질문을 정규화하고 허용한 문서 ID와 쪽만 통과시킵니다.

**이 셀의 코드 읽기.** `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다. `assert`는 조건이 참이면 출력 없이 통과하고 거짓이면 실행을 중단하므로, 바로 앞 결과를 검증하는 역할을 합니다.


In [5]:
print("질문 글자 수:", len(question))
print("AIMessage.text 있음:", bool(answer.text.strip()))
assert "[E01 p.1]" in answer.text

질문 글자 수: 15
AIMessage.text 있음: True


## 값 하나를 바꿔 비교

검색 결과에 없는 출처를 넣으면 후처리에서 거부됩니다.

**이 셀의 코드 읽기.** `try` 안의 작업이 예외를 내면 `except` 블록이 그 실패를 받아 화면에 확인 가능한 결과를 남깁니다. `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다.


In [6]:
# 검색 결과에 없는 E02는 모델 호출 뒤에도 허용하지 않습니다.
try:
    validate_answer("화요일입니다. [E02 p.1]", {("E01", 1)})
except ValueError as error:
    print(type(error).__name__, error)

ValueError 출처 허용 목록을 확인하세요.


## 수정

모델 답의 출처를 검색 결과의 허용 목록과 대조합니다.

**이 셀의 코드 읽기.** `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다. `assert`는 조건이 참이면 출력 없이 통과하고 거짓이면 실행을 중단하므로, 바로 앞 결과를 검증하는 역할을 합니다.


In [7]:
citations = validate_answer(answer.text, {("E01", 1)})
assert citations == ["[E01 p.1]"]
print("허용 목록 검증 통과")

허용 목록 검증 통과
